# GCN 中的归纳偏置：一个谱视角

__作者：[Marc Lelarge](https://www.di.ens.fr/~lelarge/)，[代码](https://github.com/dataflowr/notebooks/blob/master/graphs/GCN_inductivebias_spectral.ipynb)，课程：[dataflowr](https://dataflowr.github.io/website/)__

这里我们关注 Kipf 和 Welling 在论文 [Semi-Supervised Classification with Graph Convolutional Networks](https://arxiv.org/abs/1609.02907) 中提出的图卷积网络（GCN）。
GCN 层是最简单的图神经网络层之一，定义如下：
\begin{equation}
\label{eq:gcn_layer} h_i^{(\ell+1)} = \frac{1}{d_i+1}h_i^{(\ell)}W^{(\ell)} + \sum_{j\sim i} \frac{h_j^{(\ell)}W^{(\ell)}}{\sqrt{(d_i+1)(d_j+1)}},
\end{equation}
其中 $i\sim j$ 表示节点 $i$ 和 $j$ 在图 $G$ 中是邻居，$d_i$ 和 $d_j$ 分别是节点 $i$ 和 $j$ 的度数（也就是它们在图中的邻居数量），$h_i^{(\ell)}$ 是节点 $i$ 在第 $\ell$ 层的 embedding 表示，$W^{(\ell)}$ 是形状为 `[size_input_feature, size_output_feature]` 的可训练权重矩阵。

学习算法的[归纳偏置](https://en.wikipedia.org/wiki/Inductive_bias)是学习者用来预测未见过的输入输出的那组假设。对 GCN，我们认为归纳偏置可以表述为算法的一个简单谱性质：GCN 起低通滤波器的作用。这一论断来自近期的工作：Wu、Souza、Zhang、Fifty、Yu、Weinberger 的 [Simplifying Graph Convolutional Networks](http://proceedings.mlr.press/v97/wu19e.html) 和 NT、Maehara 的 [Revisiting Graph Neural Networks: All We Have is Low-Pass Filters](https://arxiv.org/abs/1905.09550)。

这里我们将研究一个非常简单的例子，把 GCN 的归纳偏置和图（的拉普拉斯矩阵）Fiedler 向量的性质联系起来。更一般的设定会在后续文章中讨论。


## 记号

我们考虑无向图 $G=(V,E)$，有 $n$ 个顶点，记为 $i,j \in [n]$。$i\sim j$ 表示节点 $i$ 和 $j$ 在 $G$ 中是邻居，也就是 $\{i,j\}\in E$。我们用 $A$ 表示它的[邻接矩阵](https://en.wikipedia.org/wiki/Adjacency_matrix)，用 $D$ 表示度数的对角矩阵。度数向量记为 $d$，因此 $d= A1$。向量 $x\in \mathbb{R}^n$ 的分量记为 $x_i$，但有时把向量 $x$ 看作从 $V$ 到 $\mathbb{R}$ 的函数会更方便，用 $x(i)$ 代替 $x_i$。


In [ ]:
# 安装所需的包。
!pip install -q torch-scatter -f https://pytorch-geometric.com/whl/torch-1.8.0+cu101.html
!pip install -q torch-sparse -f https://pytorch-geometric.com/whl/torch-1.8.0+cu101.html
!pip install -q torch-geometric

In [ ]:
# 可视化辅助函数。
%matplotlib inline
import torch
import networkx as nx
import matplotlib.pyplot as plt
import numpy as np

from sklearn.metrics.cluster import normalized_mutual_info_score


def visualize(h, color, cmap="Set1"):
    plt.figure(figsize=(7,7))
    plt.xticks([])
    plt.yticks([])

    if torch.is_tensor(h):
        h = h.detach().cpu().numpy()
        plt.scatter(h[:, 0], h[:, 1], s=140, c=color, cmap=cmap)
        [m0,m1] = np.median(h,axis=0)
        [min0, min1] = np.min(h,axis=0)
        [max0, max1] = np.max(h,axis=0)
        plt.vlines(m0,min1,max1)
        plt.hlines(m1,min0,max0)
        for i in range(h.shape[0]):
            plt.text(h[i,0], h[i,1], str(i))

        
    else:
        nx.draw_networkx(G, pos=nx.spring_layout(G, seed=42), with_labels=True,
                         node_color=color, cmap=cmap)
    plt.show()

## 空手道俱乐部中的社区检测

我们先从一个无监督问题开始：给定一个图，把它的节点划分成社区。在这个例子里，我们假设人们倾向于和与自己相似的人交往和结盟，这被称为[同质性（homophily）](https://en.wikipedia.org/wiki/Homophily)。

为了研究这个问题，我们关注 [Zachary 空手道俱乐部](https://en.wikipedia.org/wiki/Zachary%27s_karate_club)，并尝试从连接图恢复俱乐部的分裂。[pytorch-geometric](https://pytorch-geometric.readthedocs.io/en/latest/#) 库会非常方便。

注意，GCN 并不适合无监督设定，因为顶点没有任何标签就无法学习。不过这里不是问题，因为我们根本不会训练 GCN！在更实际的设定中，GCN 用于半监督场景：只有少数节点的标签被揭示（更多内容见 Cora 数据集那一节）。


In [ ]:
from torch_geometric.datasets import KarateClub

dataset = KarateClub()
print(f'Dataset: {dataset}:')
print('======================')
print(f'Number of graphs: {len(dataset)}')
print(f'Number of features: {dataset.num_features}')
print(f'Number of classes: {dataset.num_classes}')

如上所示，pytorch-geometric 里默认的类别（即子群）数是 4，为了简单，我们只关注分成两组的划分：


In [ ]:
data = dataset[0] 
biclasses = [int(b) for b in ((data.y == data.y[0]) + (data.y==data.y[5]))]

我们将用 [networkx](https://networkx.org/) 绘制图。在下面的图中，每个节点的颜色由它的"真实"类别决定。


In [ ]:
from torch_geometric.utils import to_networkx

G = to_networkx(data, to_undirected=True)
visualize(G, color=biclasses)

In [ ]:
def acc(predictions, classes):
    n_tot = len(classes)
    acc = np.sum([int(pred)==cla for pred,cla in zip(predictions,classes)])
    return max(acc, n_tot-acc), n_tot

[Kernighan–Lin 算法](https://en.wikipedia.org/wiki/Kernighan%E2%80%93Lin_algorithm) 是一种寻找图划分的启发式算法，下面的结果说明它很好地捕捉了我们的同质性假设。确实，该算法试图最小化两个社区之间的跨边数量。


In [ ]:
c1,c2 = nx.algorithms.community.kernighan_lin_bisection(G)
classes_kl = [0 if i in c1 else 1 for i in range(34)]
visualize(G, color=classes_kl, cmap="Set2")

In [ ]:
acc(classes_kl, biclasses)

In [ ]:
n_simu = 1000
all_acc = np.zeros(n_simu)
for i in range(n_simu):
    c1,c2 = nx.algorithms.community.kernighan_lin_bisection(G)
    classes_kl = [0 if i in c1 else 1 for i in range(34)]
    all_acc[i],_ = acc(classes_kl, biclasses)

这个算法不是确定性的，但如下所示，只有在很小一部分试验里表现不佳：


In [ ]:
bin_list = range(17,35)
_ = plt.hist(all_acc, bins=bin_list,rwidth=0.8) 

## GCN 的归纳偏置

为了演示 GCN 架构的归纳偏置，我们考虑一个简单的 3 层 GCN，并考察它不做任何训练时的表现。更准确地说，GCN 以图为输入，为每个节点 $i$ 输出一个向量 $(x_i,y_i)\in \mathbb{R}^2$。


In [ ]:
import torch
from torch.nn import Linear
from torch_geometric.nn import GCNConv
import torch.nn.functional as F

class GCN(torch.nn.Module):
    def __init__(self):
        super(GCN, self).__init__()
        self.conv1 = GCNConv(data.num_nodes, 4)# self.conv1 = GCNConv(data.num_nodes, 4)# self.conv1 = GCNConv(data.num_nodes, 4)# 没有特征……
        self.conv2 = GCNConv(4, 4)
        self.conv3 = GCNConv(4, 2)

    def forward(self, x, edge_index):
        h = self.conv1(x, edge_index)
        h = h.tanh()
        h = self.conv2(h, edge_index)
        h = h.tanh()
        h = self.conv3(h, edge_index)
        return h

torch.manual_seed(12345)
model = GCN()
print(model)

下面，我们画出图中所有节点 $i$ 的点 $(x_i,y_i)$。竖直和水平的线分别是 $x_i$ 和 $y_i$ 的中位数。颜色是真实类别。可以看到，__不需要任何学习__，这些点几乎已经按社区分离在左下角和右上角！


In [ ]:
h = model(data.x, data.edge_index)
visualize(h, color=biclasses)

In [ ]:
def color_from_vec(vec,m=None):
    if torch.is_tensor(vec):
        vec = vec.detach().cpu().numpy()
    if not(m):
        m = np.median(vec,axis=0)
    return np.array(vec < m)

注意，上面画中位数，实际上强制了图的一个均衡划分。下面我们画原始图，节点 $i$ 的颜色取决于 $x_i$ 大于还是小于中位数。


In [ ]:
color_out = color_from_vec(h[:,0])
visualize(G, color=color_out, cmap="Set2")

不做任何训练，我们只犯了很少的错误！


In [ ]:
acc(color_out,biclasses)

我们的结果可能依赖于特定的初始化，所以下面再跑几个实验：


In [ ]:
all_acc = np.zeros(n_simu)
for i in range(n_simu):
    model = GCN()
    h = model(data.x, data.edge_index)
    color_out = color_from_vec(h[:,0])
    all_acc[i],_ = acc(color_out,biclasses)

In [ ]:
_ = plt.hist(all_acc, bins=bin_list,rwidth=0.8) 

In [ ]:
np.mean(all_acc)

可以看到，平均准确率超过 $24/34$，远好于碰运气！

我们现在解释为什么随机初始化的 GCN 架构能取得这么好的结果。


## GCN 的谱分析

我们先把方程 \eqref{eq:gcn_layer} 改写成矩阵形式：
$$
h^{(\ell+1)} = S h^{(\ell)}W^{(\ell)} ,
$$
其中缩放邻接矩阵 $S\in\mathbb{R}^{n\times n}$ 定义为：如果 $i\sim j$ 或 $i=j$，则 $S_{ij} = \frac{1}{\sqrt{(d_i+1)(d_j+1)}}$，否则 $S_{ij}=0$；$h^{(\ell)}\in \mathbb{R}^{n\times f^{(\ell)}}$ 是第 $\ell$ 层节点的 embedding 表示，$W^{(\ell)}$ 是 $\mathbb{R}^{f^{(\ell)}\times f^{(\ell+1)}}$ 中的可学习权重矩阵。

为了简化，我们现在忽略上面 GCN 里的 $tanh$ 非线性，于是得到
$$
y =  S^3 W^{(1)}W^{(2)}W^{(3)},
$$
其中 $W^{(1)}\in \mathbb{R}^{n,4}$，$W^{(2)}\in \mathbb{R}^{4,4}$，$W^{(3)}\in \mathbb{R}^{4,2}$，$y\in \mathbb{R}^{n\times 2}$ 是网络的输出（注意这里 `data.x` 是单位矩阵）。
向量 $W^{(1)}W^{(2)}W^{(3)}\in \mathbb{R}^{n\times 2}$ 是一个没有特定结构的随机向量，所以要理解 GCN 的归纳偏置，我们需要理解矩阵 $S^3$ 的作用。

矩阵 $S$ 是对称的，特征值为 $\nu_1\geq \nu_2\geq ...$，对应的特征向量为 $U_1,U_2,...$。
通过 Perron-Frobenius 定理可以证明 $1=\nu_1>\nu_2\geq ...\geq \nu_n\geq -1$。下面演示这一点。


In [ ]:
from numpy import linalg as LA

A = nx.adjacency_matrix(G).todense()
A_l = A + np.eye(A.shape[0],dtype=int)
deg_l = np.dot(A_l,np.ones(A.shape[0]))
scaling = np.dot(np.transpose(1/np.sqrt(deg_l)),(1/np.sqrt(deg_l)))
S = np.multiply(scaling,A_l)
eigen_values, eigen_vectors = LA.eigh(S)

_ = plt.hist(eigen_values, bins = 40)

但对我们来说最有趣的事实与第二大特征值对应的特征向量 $U_2$ 有关，它也叫 [Fiedler 向量](https://en.wikipedia.org/wiki/Algebraic_connectivity)。

Fiedler 的一个结果表明：$G$ 在顶点集合 $\{i: U_2(i)\geq 0\}$ 上诱导的子图是连通的。这就是著名的 Fiedler 节点域定理（见 Daniel Spielman 的 [Spectral and Algebraic Graph Theory](http://cs-www.cs.yale.edu/homes/spielman/sagt/) 第 24 章）。下面我们在 $U_2$ 和 $-U_2$ 上都验证这个事实，于是这里我们把图分成了 2 个连通图（因为我们没有 $U_2(i)=0$ 的节点）。


In [ ]:
fiedler = np.array(eigen_vectors[:,-2]).squeeze()
H1 = G.subgraph([i for (i,f) in enumerate(fiedler) if f>=0])
H2 = G.subgraph([i for (i,f) in enumerate(fiedler) if -f>=0])
H = nx.union(H1,H2)
plt.figure(figsize=(7,7))
plt.xticks([])
plt.yticks([])
nx.draw_networkx(H, pos=nx.spring_layout(G, seed=42), with_labels=True)

把图分成 2 个连通图有很多种可能，而这里我们看到 Fiedler 向量给出的划分非常特殊，几乎精确对应真实的社区！


In [ ]:
visualize(G, color=[fiedler>=0], cmap="Set2")

Fiedler 向量实际上只犯了非常少的错误。另一种观察 Fiedler 向量表现的方式是把它的分量排序，并按社区标签给每个点着色，如下所示：


In [ ]:
fiedler_c = np.sort([biclasses,fiedler], axis=1)
fiedler_1 = [v for (c,v) in np.transpose(fiedler_c) if c==1]
l1 = len(fiedler_1)
fiedler_0 = [v for (c,v) in np.transpose(fiedler_c) if c==0]
l0 = len(fiedler_0)
plt.plot(range(l0),fiedler_0,'o',color='red')
plt.plot(range(l0,l1+l0),fiedler_1,'o',color='grey')
plt.plot([0]*35);

要理解为什么 Fiedler 向量的划分这么好，需要一点微积分。为简单起见，我们对矩阵 $S$ 做一个小修改：如果 $i\sim j$ 或 $i=j$，定义 $S_{ij} = \frac{1}{\sqrt{d_i d_j}}$，否则 $S_{ij}=0$。

定义（归一化的）拉普拉斯矩阵 $L=Id-S$，这样 $L$ 的特征值是 $\lambda_i=1-\nu_i$，特征向量与 $S$ 相同，都是 $U_i$。我们还定义组合[拉普拉斯矩阵](https://en.wikipedia.org/wiki/Laplacian_matrix) $L^* = D-A$。

于是我们有
\begin{equation}
\frac{x^TLx}{x^Tx} = \frac{x^TD^{-1/2}L^* D^{-1/2}x}{x^Tx}\\
= \frac{y^T L^* y}{y^TDy},
\end{equation}
其中 $y = D^{-1/2}x$。特别地，我们得到：
\begin{equation}
\lambda_2 = 1-\nu_2 = \min_{x\perp U_1}\frac{x^TLx}{x^Tx}\\
= \min_{y\perp d} \frac{y^T L^* y}{y^TDy},
\end{equation}
其中 $d$ 是度数向量。

改写最后一个方程，得到
\begin{equation}
\label{eq:minlambda}\lambda_2 = \min \frac{\sum_{i\sim j}\left(y(i)-y(j)\right)^2}{\sum_i d_i y(i)^2},
\end{equation}
其中最小值在满足 $\sum_i d_i y_i =0$ 的向量 $y$ 上取。

现在如果 $y^*$ 是达到最小值的向量，那么（差一个符号地）$U_2 =  \frac{D^{1/2}y^*}{\|D^{1/2}y^*\|}$ 就是 Fiedler 向量。特别地，$U_2$ 各元素的符号与 $y^*$ 各元素的符号相同。

要对 \eqref{eq:minlambda} 建立直觉，考虑同样的最小化问题，但加上约束 $y(i) \in \{-1,1\}$，其中 $y(i)=1$ 表示节点 $i$ 在社区 $0$，$y(i)=-1$ 表示节点 $i$ 在社区 $1$。这种情况下可以看到，分子 $\sum_{i\sim j}\left(y(i)-y(j)\right)^2$ 是两个社区之间边数的 4 倍，分母 $\sum_i d_i y(i)^2$ 是图中总边数的 2 倍。因此这个最小化问题现在变成一个组合问题：在图 $G$ 上找一个划分 $(P_1,P_2)$，满足约束 $\sum_{i\in P_1}d_i= \sum_{j\in P_2} d_j$。最后一个条件其实就是说，$G$ 在 $P_1$ 上诱导的图的边数应该等于 $G$ 在 $P_2$ 上诱导的图的边数（注意这个条件不一定有解）。因此，定义 \eqref{eq:minlambda} 中 $y^*$ 的最小化问题可以看作这个[二分问题](https://en.wikipedia.org/wiki/Graph_partition#Spectral_partitioning_and_spectral_bisection)的一个松弛。于是我们可以预期 Fiedler 向量接近 $y^*$，至少元素符号接近，这就解释了为什么 Fiedler 向量得到的划分是均衡的、且割很小，正好对应我们这里的目标。

现在既然理解了 Fiedler 向量，我们准备回到 GCN。首先，我们验证一下我们做的小简化（去掉非线性……）真的无关紧要：


In [ ]:
torch.manual_seed(12345)
model = GCN()
W1 = model.conv1.weight.detach().numpy()
W2 = model.conv2.weight.detach().numpy()
W3 = model.conv3.weight.detach().numpy()

iteration = S**3*W1*W2*W3
visualize(torch.tensor(iteration), color=biclasses)

OK，我们得到了和未训练网络一样的 embedding，但现在输出的数学公式更简单：
$$
[Y_1,Y_2] = S^3 [R_1, R_2],
$$
其中 $R_1,R_2$ 是 $\mathbb{R}^n$ 随机向量，$Y_1, Y_2$ 是上面散点图用的输出向量。

但我们可以把 $S$ 改写成 $S = \sum_{i}\nu_i U_i U_i^T$，于是 $S^3 = \sum_{i}\nu_i^3 U_i U_i^T \approx U_1U_1^T + \nu_2^3 U_2U_2^T$，因为所有其他 $\nu_i^3<< \nu_2^3$。因此得到
\begin{equation}
Y_1 \approx U_1^T R_1 U_1 + \nu_2^3 U_2^T R_1 U_2 \\
Y_2 \approx U_1^T R_2 U_1 + \nu_2^3 U_2^T R_2 U_2
\end{equation}
回忆一下，关于社区的信号就在 $U_2$ 向量里，所以可以更明确地改写为
\begin{equation}
Y_1(i) \approx a_1 + b_1 U_2(i)\\
Y_2(i) \approx a_2 + b_2 U_2(i),
\end{equation}
其中 $a_1,a_2,b_1,b_2$ 是同数量级的随机数。换句话说，点 $(Y_1(i), Y_2(i))$ 应该近似排列在一条直线上，而对应线段的两个端点对应两个社区：$U_2(i)\geq 0$ 或 $U_2(i)\leq 0$。


In [ ]:
from sklearn import linear_model
from sklearn.metrics import mean_squared_error
regr = linear_model.LinearRegression()
regr.fit(iteration[:,0].reshape(-1, 1), iteration[:,1])
plt.figure(figsize=(7,7))
plt.xticks([])
plt.yticks([])
h = np.array(iteration)
plt.scatter(h[:, 0], h[:, 1], s=140, c=biclasses, cmap="Set1")
plt.plot(h[:, 0],regr.predict(iteration[:,0].reshape(-1, 1)))

In [ ]:
def glorot_normal(in_c,out_c):
    sigma = np.sqrt(2/(in_c+out_c))
    return sigma*np.random.randn(in_c,out_c)


In [ ]:
coef = np.zeros(n_simu)
base =  np.zeros(n_simu)
for i in range(n_simu):
    iteration = glorot_normal(34,4)@glorot_normal(4,4)@glorot_normal(4,2)
    regr.fit(iteration[:,0].reshape(-1, 1), iteration[:,1])
    base[i] = mean_squared_error(iteration[:,1],regr.predict(iteration[:,0].reshape(-1, 1)))
    iteration = np.array(S**3) @ iteration
    regr.fit(iteration[:,0].reshape(-1, 1), iteration[:,1])
    coef[i] = mean_squared_error(iteration[:,1],regr.predict(iteration[:,0].reshape(-1, 1)))

下面，我们跑几次模拟，计算各点与最佳拟合直线之间的均方误差：蓝色是随机输入 $[R_1,R_2]$，橙色是输出 $[Y_1, Y_2]$（橙色几乎看不见，因为误差小得多）。我们的理论似乎得到了很好的验证 ;-)


In [ ]:
_ = plt.hist(base, bins = 34)
_ = plt.hist(coef, bins = 34)

这里我们研究了一个非常简单的例子，但更一般的结论是可能的，我们会在后续文章里看到。把关于 Fiedler 向量的分析推广需要一点谱图论，正如谱图神经网络模块里所解释的，见[图上的深度学习（2）](https://dataflowr.github.io/website/modules/graph2/)

关注[推特](https://twitter.com/marc_lelarge)！

## 感谢阅读！
